# Build a 50/50 ORB camera-pose dataset from checked training frames

This notebook treats `data/training_frames/training_frames` as a **read-only source during
each ORB build**. The retained source images must already use one consistent
orientation.

For every non-empty checked clip folder in the three source-managed classes:

- Read every retained JPG, JPEG, or PNG source image, with no per-clip limit.
- Copy each retained image unchanged into a separate derived dataset.
- Create one ORB-transformed copy in the opposite known camera position.
- Keep the exactly 50/50 rows under the same `clip_id` / `split_group`.

The `normal` source images required a one-time 180° correction on 24 July 2026.
`fallen_before_entry` and `fallen_in_view` retain their original orientation. After
that correction, the source-managed classes use the same unrotated date-banded ORB references
and the same June-to-July camera-position transform.

RANSAC match, inlier and movement checks must accept the transform before derived
images are used. Exposed border pixels are filled from the destination pose's
reference-background estimate; reflected padding is not used.
Directly prepared classes such as `no_bottle` are preserved when the source-managed classes are rebuilt.


In [ ]:
import re
import shutil
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from IPython.display import Image as DisplayImage, display
from tqdm.auto import tqdm

# Find the repository whether Jupyter starts in classifier/ or the repository root.
working_directory = Path.cwd().resolve()
project_candidates = [working_directory, *working_directory.parents]
project_root = next(
    candidate
    for candidate in project_candidates
    if (candidate / "VideoModule").is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Reuse VideoModule for folder creation, image loading, rotation and warping.
from VideoModule.io import ensure_dir, read_images
from VideoModule.preprocessing import apply_rotation
from VideoModule.preprocessing.frame_stabilization import apply_transform

RUN_ORB_DATASET_BUILD = True
REBUILD_DERIVED_CLIPS = True
ROTATE_ORB_REFERENCES_180 = False
ACTIVE_CLASSES = (
    "fallen_before_entry",
    "normal",
    "fallen_in_view",
)
SOURCE_IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")
JPEG_QUALITY = 95
PNG_COMPRESSION = 3

# ORB and RANSAC acceptance thresholds.
ORB_REFERENCE_FILES = (
    "upright.png",
    "fallen_forward_entry.png",
    "falling.png",
)
ORB_STATIC_PIXEL_RANGE = 40
ORB_FEATURE_COUNT = 5000
ORB_RATIO_TEST = 0.75
ORB_RANSAC_REPROJECTION_PIXELS = 3.0
ORB_MIN_MATCHES = 80
ORB_MIN_INLIERS = 50
ORB_MIN_INLIER_RATIO = 0.25
ORB_MAX_ROTATION_DEGREES = 20.0
ORB_MIN_SCALE = 0.85
ORB_MAX_SCALE = 1.15
ORB_MAX_TRANSLATION_FRACTION = 0.35

pipeline_root = project_root / "stoppage_detection_and_classification"
cnn_classifier_dir = pipeline_root / "cnn_classifier"
cnn_data_dir = cnn_classifier_dir / "data"
reference_root = cnn_data_dir / "ncc_reference_frames"

# These corrected source files remain read-only during every ORB build.
source_root = cnn_data_dir / "training_frames"
source_manifest_path = source_root / "training_frames_50_manifest.csv"

# All generated files go into this separate derived dataset.
orb_output_root = cnn_classifier_dir / "output" / "01_prepare_frames" / "training_frames_50"
orb_manifest_path = orb_output_root / "training_frames_50_manifest.csv"
orb_error_log_path = orb_output_root / "training_frames_50_errors.csv"

reference_date_bands = [
    {
        "name": "2026-06-18_to_2026-06-20",
        "pose_name": "june",
        "start": pd.Timestamp("2026-06-18 00:00:00"),
        "end_exclusive": pd.Timestamp("2026-06-21 00:00:00"),
        "folder": reference_root / "ncc_references_26_06_18-26_06_20",
    },
    {
        "name": "2026-07-01_to_2026-07-02",
        "pose_name": "july",
        "start": pd.Timestamp("2026-07-01 00:00:00"),
        "end_exclusive": pd.Timestamp("2026-07-03 00:00:00"),
        "folder": reference_root / "ncc_references_26_07_01-26_07_02",
    },
]

# Target the camera timestamp rather than a leading vlc-record timestamp.
camera_timestamp_pattern = re.compile(
    r"cortexvpu-[^_]+_(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}_\d{6})"
)
frame_filename_pattern = re.compile(
    r"rank_(\d+)__frame_(\d+)"
)


def filesystem_path(path):
    """Return an extended Windows path when the absolute path may exceed MAX_PATH."""
    resolved_path = Path(path).resolve()
    if sys.platform == "win32" and not str(resolved_path).startswith("\\\\?\\"):
        return Path(f"\\\\?\\{resolved_path}")
    return resolved_path

orb_manifest_columns = [
    "created_at_utc",
    "class_name",
    "clip_id",
    "split_group",
    "video_path",
    "video_name",
    "video_timestamp",
    "source_band",
    "source_pose",
    "image_kind",
    "is_orb_transformed",
    "transform_direction",
    "source_rank",
    "source_frame_index",
    "source_image_path",
    "output_image_path",
]
orb_error_columns = [
    "created_at_utc",
    "class_name",
    "clip_id",
    "video_path",
    "video_name",
    "error",
]

# Validate paths before reading any source data.
if not source_root.is_dir():
    raise FileNotFoundError(f"Checked training-frame folder does not exist: {source_root}")
if not source_manifest_path.is_file():
    raise FileNotFoundError(
        f"Checked training-frame manifest does not exist: {source_manifest_path}"
    )
for band in reference_date_bands:
    if not band["folder"].is_dir():
        raise FileNotFoundError(
            f"Reference date-band folder does not exist: {band['folder']}"
        )
ensure_dir(orb_output_root)


def parse_camera_timestamp(video_name):
    """Read the camera timestamp, including VLC-prefixed names."""
    timestamp_match = camera_timestamp_pattern.search(Path(video_name).name)
    if timestamp_match is None:
        raise ValueError(f"No camera timestamp found in: {video_name}")
    return pd.to_datetime(
        timestamp_match.group(1),
        format="%Y-%m-%d_%H-%M-%S_%f",
    )


def reference_band_for_timestamp(video_timestamp):
    """Resolve exactly one configured camera-position band."""
    matching_bands = [
        band
        for band in reference_date_bands
        if band["start"] <= video_timestamp < band["end_exclusive"]
    ]
    if len(matching_bands) != 1:
        raise ValueError(
            f"Expected one reference band for {video_timestamp}, "
            f"found {len(matching_bands)}"
        )
    return matching_bands[0]


def read_named_reference_images(reference_folder):
    """Load named PNG references through VideoModule and retain their paths."""
    reference_paths = sorted(reference_folder.glob("*.png"))
    reference_images = read_images(
        reference_folder,
        extensions=(".png",),
    )
    if len(reference_paths) != len(reference_images):
        raise ValueError(
            f"One or more references could not be read from {reference_folder}"
        )

    images_by_name = {
        image_path.name: image
        for image_path, image in zip(reference_paths, reference_images)
    }
    missing_names = set(ORB_REFERENCE_FILES) - set(images_by_name)
    if missing_names:
        raise FileNotFoundError(
            f"Missing ORB references in {reference_folder}: "
            f"{sorted(missing_names)}"
        )
    return {
        reference_name: images_by_name[reference_name]
        for reference_name in ORB_REFERENCE_FILES
    }


# Load all class-independent camera-pose references.
reference_images_by_pose = {
    band["pose_name"]: read_named_reference_images(band["folder"])
    for band in reference_date_bands
}

# Match the reference coordinate system to the corrected source-image orientation.
if ROTATE_ORB_REFERENCES_180:
    reference_images_by_pose = {
        pose_name: {
            reference_name: apply_rotation(reference_image, 180.0)
            for reference_name, reference_image in pose_images.items()
        }
        for pose_name, pose_images in reference_images_by_pose.items()
    }

# Use the existing manifest only to identify clips and their checked source folders.
source_manifest_df = pd.read_csv(source_manifest_path)
required_source_columns = {
    "class_name",
    "video_path",
    "video_name",
    "output_image_path",
}
missing_source_columns = required_source_columns - set(source_manifest_df.columns)
if missing_source_columns:
    raise ValueError(
        "Source manifest is missing columns: "
        f"{sorted(missing_source_columns)}"
    )

source_manifest_df = source_manifest_df[
    source_manifest_df["class_name"].isin(ACTIVE_CLASSES)
].copy()
if source_manifest_df.empty:
    raise ValueError("No requested classes were found in the training-frame manifest")

# Build a manifest lookup, but let the manually checked folders decide what exists.
manifest_record_by_key = {}
for (class_name, video_path), clip_rows in source_manifest_df.groupby(
    ["class_name", "video_path"],
    sort=True,
):
    video_names = clip_rows["video_name"].dropna().astype(str).unique()
    if len(video_names) != 1:
        raise ValueError(
            f"Expected one video name for {video_path}, found {video_names}"
        )
    clip_key = (class_name, Path(video_path).stem)
    if clip_key in manifest_record_by_key:
        raise ValueError(f"Duplicate source manifest clip: {clip_key}")
    manifest_record_by_key[clip_key] = {
        "video_path": str(Path(video_path).resolve()),
        "video_name": video_names[0],
    }

# Empty and manually removed folders are skipped rather than regenerated.
clip_records = []
skipped_empty_folders = []
for class_name in ACTIVE_CLASSES:
    class_folder = source_root / class_name
    if not class_folder.is_dir():
        raise FileNotFoundError(
            f"Checked class folder does not exist: {class_folder}"
        )

    for source_folder in sorted(
        path for path in class_folder.iterdir() if path.is_dir()
    ):
        original_source_folder = filesystem_path(source_folder / "original")
        checked_image_paths = sorted(
            path
            for path in original_source_folder.iterdir()
            if path.is_file()
            and path.suffix.lower() in SOURCE_IMAGE_EXTENSIONS
        )
        if not checked_image_paths:
            skipped_empty_folders.append(source_folder)
            continue
        clip_key = (class_name, source_folder.name)
        if clip_key not in manifest_record_by_key:
            raise ValueError(
                f"No source manifest record matches checked folder: {clip_key}"
            )
        manifest_record = manifest_record_by_key[clip_key]
        video_name = manifest_record["video_name"]
        video_path = manifest_record["video_path"]
        video_timestamp = parse_camera_timestamp(video_name)
        reference_band = reference_band_for_timestamp(video_timestamp)

        clip_records.append({
            "class_name": class_name,
            "clip_id": video_path,
            "split_group": video_path,
            "video_path": video_path,
            "video_name": video_name,
            "video_timestamp": video_timestamp,
            "source_band": reference_band["name"],
            "source_pose": reference_band["pose_name"],
            "source_folder": original_source_folder.resolve(),
            "source_image_count": len(checked_image_paths),
        })

if not clip_records:
    raise ValueError("No non-empty checked source clips were discovered")

# Capture source state so the review cell can prove nothing was overwritten.
source_image_paths = sorted({
    image_path.resolve()
    for record in clip_records
    for image_path in record["source_folder"].iterdir()
    if image_path.is_file()
    and image_path.suffix.lower() in SOURCE_IMAGE_EXTENSIONS
})
source_file_state_before = {
    str(image_path): (
        image_path.stat().st_size,
        image_path.stat().st_mtime_ns,
    )
    for image_path in source_image_paths
}

clip_table = pd.DataFrame(clip_records)
print(f"Checked source clips:       {len(clip_records)}")
print(f"Checked source images:      {len(source_image_paths)}")
print("Derived images per clip:    2 × retained checked images")
print(f"Empty folders skipped:      {len(skipped_empty_folders)}")
print(f"Read-only source:           {source_root}")
print(f"Derived output:             {orb_output_root}")
display(
    clip_table.groupby(
        ["class_name", "source_band", "source_pose"]
    ).size().rename("clips")
)

In [ ]:
def append_csv_rows(rows, csv_path, columns):
    """Append fixed-schema rows after a complete clip has been written."""
    if not rows:
        return

    csv_exists = csv_path.exists() and csv_path.stat().st_size > 0
    if csv_exists:
        existing_columns = list(pd.read_csv(csv_path, nrows=0).columns)
        if existing_columns != columns:
            raise ValueError(
                f"CSV schema mismatch for {csv_path}. "
                f"Expected {columns}, found {existing_columns}."
            )

    pd.DataFrame(rows, columns=columns).to_csv(
        csv_path,
        mode="a",
        header=not csv_exists,
        index=False,
    )


def read_checked_clip_images(source_folder):
    """Read every retained checked JPEG using VideoModule."""
    image_paths = sorted(
        path
        for path in source_folder.iterdir()
        if path.is_file()
        and path.suffix.lower() in SOURCE_IMAGE_EXTENSIONS
    )
    images = read_images(
        source_folder,
        extensions=SOURCE_IMAGE_EXTENSIONS,
    )
    if len(image_paths) != len(images):
        raise ValueError(
            f"One or more checked images could not be read: {source_folder}"
        )
    if not image_paths:
        raise ValueError(f"No checked images remain in {source_folder}")
    return image_paths, images


def source_frame_metadata(image_path, fallback_rank):
    """Recover the original NCC rank and frame index from the checked filename."""
    filename_match = frame_filename_pattern.search(image_path.name)
    if filename_match is None:
        return int(fallback_rank), np.nan
    return int(filename_match.group(1)), int(filename_match.group(2))


def build_static_background_mask(images):
    """Keep pixels that stay stable across all class-specific references."""
    grayscale_images = [
        cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(np.int16)
        for image in images
    ]
    expected_shape = grayscale_images[0].shape
    if any(image.shape != expected_shape for image in grayscale_images):
        raise ValueError("ORB references must all have the same dimensions")

    # People change across the references; the static camera background does not.
    grayscale_stack = np.stack(grayscale_images, axis=0)
    pixel_range = np.ptp(grayscale_stack, axis=0)
    static_mask = (pixel_range <= ORB_STATIC_PIXEL_RANGE).astype(np.uint8) * 255

    # Remove isolated pixels and shrink boundaries around foreground movement.
    static_mask = cv2.morphologyEx(
        static_mask,
        cv2.MORPH_OPEN,
        np.ones((5, 5), dtype=np.uint8),
    )
    static_mask = cv2.erode(
        static_mask,
        np.ones((7, 7), dtype=np.uint8),
    )
    static_fraction = float(np.mean(static_mask > 0))
    if static_fraction < 0.20:
        raise ValueError(
            f"Static-background mask is too small: {static_fraction:.1%}"
        )
    return static_mask, static_fraction


def affine_transform_stats(affine_matrix, image_shape, match_count, inlier_mask):
    """Return readable ORB/RANSAC quality and movement measurements."""
    height, width = image_shape[:2]
    scale = float(np.hypot(affine_matrix[0, 0], affine_matrix[1, 0]))
    rotation_degrees = float(
        np.degrees(
            np.arctan2(affine_matrix[1, 0], affine_matrix[0, 0])
        )
    )
    translation_x = float(affine_matrix[0, 2])
    translation_y = float(affine_matrix[1, 2])
    inlier_count = int(inlier_mask.sum())

    return {
        "match_count": int(match_count),
        "inlier_count": inlier_count,
        "inlier_ratio": inlier_count / int(match_count),
        "rotation_degrees": rotation_degrees,
        "scale": scale,
        "translation_x": translation_x,
        "translation_y": translation_y,
        "translation_x_fraction": abs(translation_x) / width,
        "translation_y_fraction": abs(translation_y) / height,
    }


def validate_affine_transform(stats):
    """Reject weak feature matches and implausibly large movements."""
    failures = []
    if stats["match_count"] < ORB_MIN_MATCHES:
        failures.append(
            f"matches {stats['match_count']} < {ORB_MIN_MATCHES}"
        )
    if stats["inlier_count"] < ORB_MIN_INLIERS:
        failures.append(
            f"inliers {stats['inlier_count']} < {ORB_MIN_INLIERS}"
        )
    if stats["inlier_ratio"] < ORB_MIN_INLIER_RATIO:
        failures.append(
            f"inlier ratio {stats['inlier_ratio']:.3f} < "
            f"{ORB_MIN_INLIER_RATIO:.3f}"
        )
    if abs(stats["rotation_degrees"]) > ORB_MAX_ROTATION_DEGREES:
        failures.append(
            f"rotation {stats['rotation_degrees']:.2f} degrees is too large"
        )
    if not ORB_MIN_SCALE <= stats["scale"] <= ORB_MAX_SCALE:
        failures.append(f"scale {stats['scale']:.3f} is outside limits")
    if stats["translation_x_fraction"] > ORB_MAX_TRANSLATION_FRACTION:
        failures.append("horizontal translation is outside limits")
    if stats["translation_y_fraction"] > ORB_MAX_TRANSLATION_FRACTION:
        failures.append("vertical translation is outside limits")

    if failures:
        raise ValueError("ORB transform rejected: " + "; ".join(failures))


def estimate_reference_pose_transform(source_images, target_images):
    """Estimate one background-only affine transform between camera poses."""
    source_mask, source_static_fraction = build_static_background_mask(
        source_images
    )
    target_mask, target_static_fraction = build_static_background_mask(
        target_images
    )

    orb_detector = cv2.ORB_create(nfeatures=ORB_FEATURE_COUNT)
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
    all_source_points = []
    all_target_points = []

    # Combine all reference types into one class-independent RANSAC fit.
    for source_image, target_image in zip(source_images, target_images):
        source_gray = cv2.cvtColor(source_image, cv2.COLOR_BGR2GRAY)
        target_gray = cv2.cvtColor(target_image, cv2.COLOR_BGR2GRAY)
        source_keypoints, source_descriptors = orb_detector.detectAndCompute(
            source_gray,
            source_mask,
        )
        target_keypoints, target_descriptors = orb_detector.detectAndCompute(
            target_gray,
            target_mask,
        )
        if source_descriptors is None or target_descriptors is None:
            continue

        candidate_pairs = matcher.knnMatch(
            source_descriptors,
            target_descriptors,
            k=2,
        )
        good_matches = [
            pair[0]
            for pair in candidate_pairs
            if len(pair) == 2
            and pair[0].distance < ORB_RATIO_TEST * pair[1].distance
        ]
        all_source_points.extend(
            source_keypoints[match.queryIdx].pt
            for match in good_matches
        )
        all_target_points.extend(
            target_keypoints[match.trainIdx].pt
            for match in good_matches
        )

    match_count = len(all_source_points)
    if match_count < 3:
        raise ValueError("Too few ORB matches to estimate a camera transform")

    affine_matrix, inlier_mask = cv2.estimateAffinePartial2D(
        np.float32(all_source_points),
        np.float32(all_target_points),
        method=cv2.RANSAC,
        ransacReprojThreshold=ORB_RANSAC_REPROJECTION_PIXELS,
        maxIters=10000,
        confidence=0.999,
        refineIters=50,
    )
    if affine_matrix is None or inlier_mask is None:
        raise ValueError("RANSAC could not estimate an ORB transform")

    stats = affine_transform_stats(
        affine_matrix,
        source_images[0].shape,
        match_count,
        inlier_mask,
    )
    stats["source_static_fraction"] = source_static_fraction
    stats["target_static_fraction"] = target_static_fraction
    validate_affine_transform(stats)
    return affine_matrix, stats


def build_reference_background(reference_images):
    """Estimate one destination-pose background from its reference images."""
    image_shapes = {image.shape for image in reference_images}
    if len(image_shapes) != 1:
        raise ValueError("Reference images must all have the same dimensions")

    # The per-pixel median suppresses people that move between reference images.
    stacked_images = np.stack(reference_images, axis=0).astype(np.float32)
    median_background = np.median(stacked_images, axis=0)
    return np.rint(median_background).astype(np.uint8)


def warp_to_other_camera_pose(
    image,
    affine_matrix,
    destination_background,
):
    """Warp an image and fill exposed pixels from the destination background."""
    homography_matrix = np.vstack([
        affine_matrix,
        np.array([0.0, 0.0, 1.0]),
    ])
    height, width = image.shape[:2]

    # Use VideoModule for the image warp, with empty pixels initially set to zero.
    warped_image = apply_transform(
        image,
        homography_matrix,
        output_size=(width, height),
        border_mode=cv2.BORDER_CONSTANT,
        border_value=(0, 0, 0),
    )

    # Warp a white mask to identify which destination pixels contain real source data.
    source_validity_mask = np.full((height, width), 255, dtype=np.uint8)
    warped_validity_mask = apply_transform(
        source_validity_mask,
        homography_matrix,
        output_size=(width, height),
        border_mode=cv2.BORDER_CONSTANT,
        border_value=(0, 0, 0),
    )

    # Resize only if a future reference set uses a different resolution.
    if destination_background.shape[:2] != (height, width):
        destination_background = cv2.resize(
            destination_background,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        )

    # The warped image is already premultiplied at interpolated edge pixels.
    # Add the complementary amount of real destination background for a soft seam.
    source_weight = warped_validity_mask.astype(np.float32) / 255.0
    background_weight = 1.0 - source_weight[..., np.newaxis]
    composited_image = (
        warped_image.astype(np.float32)
        + destination_background.astype(np.float32) * background_weight
    )
    return np.clip(np.rint(composited_image), 0, 255).astype(np.uint8)


def estimate_unknown_images_to_pose_transform(
    sample_images,
    target_reference_images,
):
    """Estimate one confidence-gated transform for a potential future knock."""
    if len(sample_images) < len(ORB_REFERENCE_FILES):
        raise ValueError(
            f"Provide at least {len(ORB_REFERENCE_FILES)} images "
            "from the same clip"
        )

    sample_positions = np.linspace(
        0,
        len(sample_images) - 1,
        num=len(ORB_REFERENCE_FILES),
    )
    selected_images = [
        sample_images[int(round(position))]
        for position in sample_positions
    ]

    accepted_candidates = []
    rejection_messages = []
    for pre_alignment_flip_180 in (False, True):
        oriented_images = [
            (
                apply_rotation(image, 180.0)
                if pre_alignment_flip_180
                else image
            )
            for image in selected_images
        ]
        try:
            affine_matrix, stats = estimate_reference_pose_transform(
                oriented_images,
                target_reference_images,
            )
        except ValueError as error:
            rejection_messages.append(
                f"flip_180={pre_alignment_flip_180}: {error}"
            )
            continue

        stats = dict(stats)
        stats["pre_alignment_flip_180"] = pre_alignment_flip_180
        accepted_candidates.append((affine_matrix, stats))

    if not accepted_candidates:
        raise ValueError(
            "Unknown camera position could not be aligned safely. "
            + " | ".join(rejection_messages)
        )

    return max(
        accepted_candidates,
        key=lambda candidate: (
            candidate[1]["inlier_count"],
            candidate[1]["inlier_ratio"],
        ),
    )


# Calculate one accepted forward transform and its exact inverse.
june_reference_images = [
    reference_images_by_pose["june"][reference_name]
    for reference_name in ORB_REFERENCE_FILES
]
july_reference_images = [
    reference_images_by_pose["july"][reference_name]
    for reference_name in ORB_REFERENCE_FILES
]

# Build clean border-fill images in each destination camera position.
june_pose_background = build_reference_background(june_reference_images)
july_pose_background = build_reference_background(july_reference_images)

june_to_july_affine, orb_transform_stats = estimate_reference_pose_transform(
    june_reference_images,
    july_reference_images,
)
july_to_june_affine = cv2.invertAffineTransform(june_to_july_affine)

print("Accepted June -> July background transform:")
print(june_to_july_affine)
print(pd.Series(orb_transform_stats))


In [ ]:
if not RUN_ORB_DATASET_BUILD:
    print(
        "ORB dataset build is disabled. "
        "Set RUN_ORB_DATASET_BUILD = True in the setup cell."
    )
    current_run_summary_df = pd.DataFrame()
else:
    # Rebuild only the derived manifest. The checked training-frame manifest is read-only.
    if REBUILD_DERIVED_CLIPS:
        # Preserve directly prepared classes that are not part of this source rebuild.
        if orb_manifest_path.is_file():
            existing_manifest_df = pd.read_csv(orb_manifest_path)
            preserved_manifest_df = existing_manifest_df[
                ~existing_manifest_df["class_name"].isin(ACTIVE_CLASSES)
            ].copy()
        else:
            preserved_manifest_df = pd.DataFrame(columns=orb_manifest_columns)
        preserved_manifest_df.to_csv(orb_manifest_path, index=False)
        pd.DataFrame(columns=orb_error_columns).to_csv(
            orb_error_log_path,
            index=False,
        )

        # Remove only stale derived clip folders. Checked source folders are read-only.
        retained_derived_keys = {
            (record["class_name"], Path(record["source_folder"]).name)
            for record in clip_records
        }
        for class_name in ACTIVE_CLASSES:
            derived_class_folder = ensure_dir(orb_output_root / class_name)
            for derived_clip_folder in derived_class_folder.iterdir():
                if not derived_clip_folder.is_dir():
                    continue
                derived_key = (class_name, derived_clip_folder.name)
                if derived_key not in retained_derived_keys:
                    shutil.rmtree(derived_clip_folder)

    current_run_summaries = []

    for clip_number, record in enumerate(
        tqdm(clip_records, desc="Checked clips"),
        start=1,
    ):
        created_at_utc = pd.Timestamp.now(tz="UTC").isoformat()
        class_name = record["class_name"]
        clip_id = record["clip_id"]
        source_folder = Path(record["source_folder"])

        try:
            source_paths, source_images = read_checked_clip_images(
                source_folder
            )

            # Select the one fixed transform for this source camera position.
            if record["source_pose"] == "june":
                pose_transform = june_to_july_affine
                destination_background = july_pose_background
                transform_direction = "june_to_july"
            elif record["source_pose"] == "july":
                pose_transform = july_to_june_affine
                destination_background = june_pose_background
                transform_direction = "july_to_june"
            else:
                raise ValueError(
                    f"Unknown source pose: {record['source_pose']}"
                )

            # All writes are constrained to the separate derived root.
            derived_clip_folder = ensure_dir(
                orb_output_root
                / class_name
                / source_folder.name
            )
            original_folder = ensure_dir(derived_clip_folder / "original")
            transformed_folder = ensure_dir(
                derived_clip_folder / "orb_transformed"
            )

            # Replace only old derived files for this clip.
            if REBUILD_DERIVED_CLIPS:
                for output_folder in (original_folder, transformed_folder):
                    for old_path in output_folder.iterdir():
                        if (
                            old_path.is_file()
                            and old_path.suffix.lower() in SOURCE_IMAGE_EXTENSIONS
                        ):
                            old_path.unlink()

            manifest_rows_for_clip = []
            for fallback_rank, (source_path, source_image) in enumerate(
                zip(source_paths, source_images),
                start=1,
            ):
                source_rank, source_frame_index = source_frame_metadata(
                    source_path,
                    fallback_rank,
                )

                # Preserve an exact byte-for-byte copy of the checked source image.
                original_output_path = original_folder / source_path.name
                shutil.copy2(source_path, original_output_path)

                # Create one copy in the opposite known camera position.
                transformed_image = warp_to_other_camera_pose(
                    source_image,
                    pose_transform,
                    destination_background,
                )
                transformed_output_path = transformed_folder / source_path.name
                if source_path.suffix.lower() in {".jpg", ".jpeg"}:
                    write_parameters = [
                        cv2.IMWRITE_JPEG_QUALITY,
                        JPEG_QUALITY,
                    ]
                else:
                    write_parameters = [
                        cv2.IMWRITE_PNG_COMPRESSION,
                        PNG_COMPRESSION,
                    ]
                write_succeeded = cv2.imwrite(
                    str(transformed_output_path),
                    transformed_image,
                    write_parameters,
                )
                if not write_succeeded:
                    raise RuntimeError(
                        f"Could not save: {transformed_output_path}"
                    )

                common_row = {
                    "created_at_utc": created_at_utc,
                    "class_name": class_name,
                    "clip_id": clip_id,
                    "split_group": record["split_group"],
                    "video_path": record["video_path"],
                    "video_name": record["video_name"],
                    "video_timestamp": record["video_timestamp"].isoformat(),
                    "source_band": record["source_band"],
                    "source_pose": record["source_pose"],
                    "source_rank": source_rank,
                    "source_frame_index": source_frame_index,
                    "source_image_path": str(source_path.resolve()),
                }
                manifest_rows_for_clip.append({
                    **common_row,
                    "image_kind": "original",
                    "is_orb_transformed": False,
                    "transform_direction": "none",
                    "output_image_path": str(
                        original_output_path.resolve()
                    ),
                })
                manifest_rows_for_clip.append({
                    **common_row,
                    "image_kind": "orb_transformed",
                    "is_orb_transformed": True,
                    "transform_direction": transform_direction,
                    "output_image_path": str(
                        transformed_output_path.resolve()
                    ),
                })

            expected_derived_count = len(source_paths) * 2
            if len(manifest_rows_for_clip) != expected_derived_count:
                raise RuntimeError(
                    f"Expected {expected_derived_count} derived rows, "
                    f"found {len(manifest_rows_for_clip)}"
                )

            append_csv_rows(
                manifest_rows_for_clip,
                orb_manifest_path,
                orb_manifest_columns,
            )
            current_run_summaries.append({
                "class_name": class_name,
                "video_name": record["video_name"],
                "source_pose": record["source_pose"],
                "transform_direction": transform_direction,
                "source_images_read": len(source_paths),
                "derived_images_written": len(manifest_rows_for_clip),
                "status": "ok",
                "error": "",
            })
            print(
                f"[{clip_number}/{len(clip_records)}] {class_name}: "
                f"{len(source_paths)} original + "
                f"{len(source_paths)} {transform_direction} copies"
            )
        except Exception as error:
            append_csv_rows(
                [{
                    "created_at_utc": created_at_utc,
                    "class_name": class_name,
                    "clip_id": clip_id,
                    "video_path": record["video_path"],
                    "video_name": record["video_name"],
                    "error": str(error),
                }],
                orb_error_log_path,
                orb_error_columns,
            )
            current_run_summaries.append({
                "class_name": class_name,
                "video_name": record["video_name"],
                "source_pose": record["source_pose"],
                "transform_direction": "",
                "source_images_read": 0,
                "derived_images_written": 0,
                "status": "failed",
                "error": str(error),
            })
            print(
                f"[{clip_number}/{len(clip_records)}] "
                f"Failed {record['video_name']}: {error}"
            )

    current_run_summary_df = pd.DataFrame(current_run_summaries)
    display(current_run_summary_df)
    print(f"Read-only source:  {source_root}")
    print(f"Derived manifest: {orb_manifest_path}")
    print(f"Derived errors:   {orb_error_log_path}")


In [ ]:
# Prove that the checked training-frame source files were not overwritten.
source_file_state_after = {
    str(image_path): (
        image_path.stat().st_size,
        image_path.stat().st_mtime_ns,
    )
    for image_path in source_image_paths
}
if source_file_state_after != source_file_state_before:
    changed_source_paths = sorted({
        *(
            path
            for path, state in source_file_state_before.items()
            if source_file_state_after.get(path) != state
        ),
        *(
            path
            for path in source_file_state_after
            if path not in source_file_state_before
        ),
    })
    raise RuntimeError(
        "Checked training-frame source files changed during this run: "
        f"{changed_source_paths[:10]}"
    )
print(
    f"Read-only source verified unchanged: "
    f"{len(source_file_state_after)} images"
)

# Confirm one original and one transformed copy for every retained source image.
if orb_manifest_path.exists() and orb_manifest_path.stat().st_size > 0:
    complete_orb_manifest_df = pd.read_csv(orb_manifest_path)
    orb_clip_summary_df = (
        complete_orb_manifest_df
        .groupby(["class_name", "clip_id", "image_kind"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    required_image_kinds = {"original", "orb_transformed"}
    missing_image_kinds = required_image_kinds - set(
        orb_clip_summary_df.columns
    )
    if missing_image_kinds:
        raise ValueError(
            f"Derived manifest is missing: {sorted(missing_image_kinds)}"
        )

    invalid_clips = orb_clip_summary_df[
        (orb_clip_summary_df["original"] < 1)
        | (
            orb_clip_summary_df["original"]
            != orb_clip_summary_df["orb_transformed"]
        )
    ]
    if not invalid_clips.empty:
        raise ValueError(
            "Some clips are not exactly 50/50 original and transformed"
        )

    display(
        complete_orb_manifest_df.groupby(
            ["class_name", "source_pose", "image_kind"]
        ).agg(
            clips=("clip_id", "nunique"),
            images=("output_image_path", "size"),
        )
    )
    print(
        f"Validated {orb_clip_summary_df['clip_id'].nunique()} non-empty "
        "clips with one transformed copy per retained original."
    )
    print(
        "All original/transformed pairs share the same clip_id and "
        "split_group."
    )

    # Show a few paired previews without touching the checked source folder.
    preview_originals = complete_orb_manifest_df[
        complete_orb_manifest_df["image_kind"] == "original"
    ].head(3)
    for _, original_row in preview_originals.iterrows():
        transformed_row = complete_orb_manifest_df[
            (
                complete_orb_manifest_df["clip_id"]
                == original_row["clip_id"]
            )
            & (
                complete_orb_manifest_df["source_rank"]
                == original_row["source_rank"]
            )
            & (
                complete_orb_manifest_df["image_kind"]
                == "orb_transformed"
            )
        ].iloc[0]
        print(
            f"{original_row['class_name']} | "
            f"{original_row['video_name']} | "
            f"{transformed_row['transform_direction']}"
        )
        display(
            DisplayImage(
                filename=original_row["output_image_path"],
                width=480,
            )
        )
        display(
            DisplayImage(
                filename=transformed_row["output_image_path"],
                width=480,
            )
        )
else:
    print("No derived ORB manifest yet. Run the build cell first.")
